In [1]:
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import numpy as np
from pyscf import gto, scf, fci
from flax import nnx
import optax
from tqdm import tqdm

# ==============================================================================
# 1. 全局参数 & H₂ 分子定义
# ==============================================================================
K = 2                      # 同时计算的态数（基态 + 第一激发态）
bond_length = 1.4
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]
mol = gto.M(atom=geometry, basis='STO-3G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)

# FCI 精确基准
cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()
print("="*60)
print("H₂ FCI 基准能量")
print("="*60)
for i, e in enumerate(E_fcis):
    exc = (e - E_fcis[0]) * 27.2114
    print(f"E{i} = {e:.8f} Ha  |  激发能: {exc:.4f} eV")

# NetKet 哈密顿量
ha = nkx.operator.from_pyscf_molecule(mol)

# 原始 Hilbert 空间（2个轨道，每个自旋1个电子）
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    n_fermions_per_spin=(1,1),
)
hi_ensemble = hi ** K          # 扩展空间

# 采样器：使用 TensorRule 组合 K 个单链规则
edges = [(0, 1), (2, 3)]
g = nk.graph.Graph(edges=edges)
single_rule = nk.sampler.rules.FermionHopRule(hilbert=hi, graph=g)
tensor_rule = nk.sampler.rules.TensorRule(hilbert=hi_ensemble, rules=[single_rule]*K)
sampler = nk.sampler.MetropolisSampler(hi_ensemble, rule=tensor_rule, n_chains=16, sweep_size=32)

# ==============================================================================
# 2. 神经网络 Ansatz
# ==============================================================================
class SingleStateAnsatz(nnx.Module):
    def __init__(self, n_spin_orbitals: int, hidden_dim=16, *, rngs: nnx.Rngs):
        super().__init__()
        self.linear1 = nnx.Linear(n_spin_orbitals, hidden_dim, rngs=rngs, param_dtype=complex)
        self.linear2 = nnx.Linear(hidden_dim, hidden_dim, rngs=rngs, param_dtype=complex)
        self.output = nnx.Linear(hidden_dim, 1, rngs=rngs, param_dtype=complex)

    def __call__(self, x):
        h = nnx.tanh(self.linear1(x.astype(complex)))
        h = nnx.tanh(self.linear2(h))
        out = self.output(h)
        return jnp.squeeze(out)

class NESTotalAnsatz(nnx.Module):
    def __init__(self, n_spin_orbitals, n_states=K, hidden_dim=16):
        super().__init__()
        self.n_states = n_states
        self.n_spin_orbitals = n_spin_orbitals

        key = jax.random.key(42)
        keys = jax.random.split(key, n_states)
        self.single_ansatz_list = [
            SingleStateAnsatz(n_spin_orbitals, hidden_dim, rngs=nnx.Rngs(keys[i]))
            for i in range(n_states)
        ]

    def __call__(self, x_batch):
        """x_batch: (K, n_spin_orbitals) -> (psi_total, M)"""
        K_state = self.n_states
        M = []
        for i in range(K_state):
            row = [self.single_ansatz_list[j](x_batch[i]) for j in range(K_state)]
            M.append(jnp.array(row))
        M = jnp.stack(M)
        psi_total = jnp.linalg.det(M)
        return psi_total, M

# ==============================================================================
# 3. 辅助函数：Hψ 与局部能量矩阵
# ==============================================================================
def Ham_psi(ha, single_ansatz, x):
    """单态 Ansatz 在组态 x 上的 Hψ 值"""
    x = x.squeeze()
    x_primes, mels = ha.get_conn_padded(x)
    psi_vals = jax.vmap(single_ansatz)(x_primes)
    return jnp.sum(mels * psi_vals)

def Ham_Psi(ha, total_ansatz, x_batch):
    """计算 HΨ 矩阵，形状 (K, K)"""
    K = total_ansatz.n_states
    H_mat = []
    for i in range(K):
        row = []
        for j in range(K):
            v = Ham_psi(ha, total_ansatz.single_ansatz_list[j], x_batch[i])
            row.append(v)
        H_mat.append(row)
    return jnp.array(H_mat, dtype=complex)

def compute_local_energy(ha, total_ansatz, x_batch, eps=1e-6):
    """
    计算局部能量矩阵及其迹
    x_batch: (K, n_spin_orbitals)
    返回 (trace_real, el_mat)
    """
    psi, M = total_ansatz(x_batch)
    # 动态正则化：若行列式接近零则增大 eps
    det_val = jnp.linalg.det(M)
    cond = jnp.abs(det_val) < 1e-4
    actual_eps = jnp.where(cond, 1e-4, eps)
    M_reg = M + actual_eps * jnp.eye(M.shape[0], dtype=M.dtype)
    Hp = Ham_Psi(ha, total_ansatz, x_batch)
    el_mat = jnp.linalg.solve(M_reg, Hp)
    return jnp.trace(el_mat).real, el_mat

compute_local_energy_batch = jax.vmap(
    compute_local_energy,
    in_axes=(None, None, 0, None),
    out_axes=(0, 0)
)

def compute_average_local_energy(ha, model, samples, eps=1e-6):
    """
    samples: (n_samples, K, n_spin_orbitals)
    """
    tr_els, el_mats = compute_local_energy_batch(ha, model, samples, eps)
    tr_avg = tr_els.mean()
    el_mat_avg = el_mats.mean(axis=0)
    return tr_avg, el_mat_avg

def loss_fn(params, ha, x_batch):
    graphdef, variables = params
    model = nnx.merge(graphdef, variables)
    tr_avg, _ = compute_average_local_energy(ha, model, x_batch, eps=1e-6)
    return tr_avg

value_and_grad = jax.value_and_grad(loss_fn, argnums=0)

# ==============================================================================
# 4. forward 函数（供采样器使用，返回 log|Ψ|）
# ==============================================================================
def forward(params, x_batch):
    """
    x_batch: (n_chains, K * n_spin_orbitals)
    返回: (n_chains,) 每个元素的 log|Ψ|
    """
    graphdef, variables = params
    model = nnx.merge(graphdef, variables)
    n_chains = x_batch.shape[0]
    K_state = model.n_states
    n_spin = model.n_spin_orbitals   # 4
    x_reshaped = x_batch.reshape(n_chains, K_state, n_spin)
    
    def single_logpsi(x):
        psi, _ = model(x)
        return jnp.log(psi)
    
    log_psi_batch = jax.vmap(single_logpsi)(x_reshaped)
    return log_psi_batch

# ==============================================================================
# 5. 初始化模型、采样器、优化器（带梯度裁剪）
# ==============================================================================
total_ansatz = NESTotalAnsatz(n_spin_orbitals=4, n_states=K, hidden_dim=16)
graphdef, variables = nnx.split(total_ansatz)

# 采样器状态（只初始化一次，训练中不 reset）
sampler_state = sampler.init_state(forward, (graphdef, variables), seed=1)

# 优化器：Adam + 全局梯度裁剪
optimizer = optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.adam(learning_rate=1e-3)
)
opt_state = optimizer.init(variables)

# ==============================================================================
# 6. 训练循环
# ==============================================================================
n_iter = 100
chain_length = 200          # 每条链采样步数
loss_record = []

print("\n开始训练 NES-VMC...")
for step in tqdm(range(n_iter)):
    # 重要：不要 reset 采样器！只使用上一次的 state 继续采样
    samples, sampler_state = sampler.sample(
        forward, (graphdef, variables), state=sampler_state, chain_length=chain_length
    )
    # samples 形状: (n_chains, chain_length, K*4)
    samples_flat = samples.reshape(-1, K, 4)   # (n_samples, K, 4)
    
    loss_val, grads = value_and_grad((graphdef, variables), ha, samples_flat)
    loss_record.append(loss_val)
    
    # 更新参数
    grad_graph, grad_vars = grads
    updates, opt_state = optimizer.update(grad_vars, opt_state, variables)
    variables = optax.apply_updates(variables, updates)
    
    if step % 5 == 0:
        print(f"Step {step:4d} | Trace of local energy matrix: {loss_val:.6f} Ha")

# 合并最终模型
total_ansatz = nnx.merge(graphdef, variables)

# ==============================================================================
# 7. 最终采样，对角化得到各态能量
# ==============================================================================
print("\n最终采样，计算能量矩阵...")
final_samples, _ = sampler.sample(
    forward, (graphdef, variables), state=sampler_state, chain_length=2000
)
final_samples_flat = final_samples.reshape(-1, K, 4)

_, el_mat_avg = compute_average_local_energy(ha, total_ansatz, final_samples_flat, eps=1e-6)
# 对称化保证 Hermitian
el_mat_sym = (el_mat_avg + el_mat_avg.conj().T) / 2
eigen_energies = jnp.linalg.eigvalsh(el_mat_sym).real
# 按能量升序排列
eigen_energies = jnp.sort(eigen_energies)

print("\n" + "="*60)
print("NES-VMC 计算得到的激发态能量 (Ha)")
print("="*60)
for i, e in enumerate(eigen_energies):
    exc = (e - eigen_energies[0]) * 27.2114
    print(f"E{i} = {e:.8f} Ha  |  激发能: {exc:.4f} eV")

/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


H₂ FCI 基准能量
E0 = -1.01546825 Ha  |  激发能: 0.0000 eV
E1 = -0.87542794 Ha  |  激发能: 3.8107 eV
E2 = -0.42938376 Ha  |  激发能: 15.9482 eV
E3 = -0.26922131 Ha  |  激发能: 20.3064 eV

开始训练 NES-VMC...


  1%|          | 1/100 [00:04<08:07,  4.93s/it]

Step    0 | Trace of local energy matrix: -1.493442 Ha


  6%|▌         | 6/100 [00:07<01:11,  1.31it/s]

...

Step   90 | Trace of local energy matrix: -1.431035 Ha


 96%|█████████▌| 96/100 [00:55<00:02,  1.93it/s]

Step   95 | Trace of local energy matrix: -1.412240 Ha


100%|██████████| 100/100 [00:57<00:00,  1.73it/s]



最终采样，计算能量矩阵...

NES-VMC 计算得到的激发态能量 (Ha)
E0 = -1.01923528 Ha  |  激发能: 0.0000 eV
E1 = -0.40251365 Ha  |  激发能: 16.7819 eV


In [2]:
# ==============================================================================
# 6. 训练循环
# ==============================================================================
n_iter = 100
chain_length = 200          # 每条链采样步数
loss_record = []

print("\n开始训练 NES-VMC...")
for step in tqdm(range(n_iter)):
    # 重要：不要 reset 采样器！只使用上一次的 state 继续采样
    samples, sampler_state = sampler.sample(
        forward, (graphdef, variables), state=sampler_state, chain_length=chain_length
    )
    # samples 形状: (n_chains, chain_length, K*4)
    samples_flat = samples.reshape(-1, K, 4)   # (n_samples, K, 4)
    
    loss_val, grads = value_and_grad((graphdef, variables), ha, samples_flat)
    loss_record.append(loss_val)
    
    # 更新参数
    grad_graph, grad_vars = grads
    updates, opt_state = optimizer.update(grad_vars, opt_state, variables)
    variables = optax.apply_updates(variables, updates)
    
    if step % 5 == 0:
        print(f"Step {step:4d} | Trace of local energy matrix: {loss_val:.6f} Ha")

# 合并最终模型
total_ansatz = nnx.merge(graphdef, variables)

# ==============================================================================
# 7. 最终采样，对角化得到各态能量
# ==============================================================================
print("\n最终采样，计算能量矩阵...")
final_samples, _ = sampler.sample(
    forward, (graphdef, variables), state=sampler_state, chain_length=2000
)
final_samples_flat = final_samples.reshape(-1, K, 4)

_, el_mat_avg = compute_average_local_energy(ha, total_ansatz, final_samples_flat, eps=1e-6)
# 对称化保证 Hermitian
el_mat_sym = (el_mat_avg + el_mat_avg.conj().T) / 2
eigen_energies = jnp.linalg.eigvalsh(el_mat_sym).real
# 按能量升序排列
eigen_energies = jnp.sort(eigen_energies)

print("\n" + "="*60)
print("NES-VMC 计算得到的激发态能量 (Ha)")
print("="*60)
for i, e in enumerate(eigen_energies):
    exc = (e - eigen_energies[0]) * 27.2114
    print(f"E{i} = {e:.8f} Ha  |  激发能: {exc:.4f} eV")


开始训练 NES-VMC...


  1%|          | 1/100 [00:00<01:16,  1.29it/s]

Step    0 | Trace of local energy matrix: -1.418756 Ha


  6%|▌         | 6/100 [00:03<00:50,  1.86it/s]

Step    5 | Trace of local energy matrix: -1.421367 Ha


...

Step   90 | Trace of local energy matrix: -1.372783 Ha


 96%|█████████▌| 96/100 [00:53<00:02,  1.94it/s]

Step   95 | Trace of local energy matrix: -1.362928 Ha


100%|██████████| 100/100 [00:56<00:00,  1.78it/s]



最终采样，计算能量矩阵...

NES-VMC 计算得到的激发态能量 (Ha)
E0 = -0.97864831 Ha  |  激发能: 0.0000 eV
E1 = -0.39162898 Ha  |  激发能: 15.9736 eV


In [ ]:
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import numpy as np
from pyscf import gto, scf, fci
from flax import nnx
import optax
from tqdm import tqdm

# ==============================================================================
# 1. 全局参数 & H₂ 分子
# ==============================================================================
K = 2
bond_length = 1.4
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]
mol = gto.M(atom=geometry, basis='STO-3G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)

# FCI 基准
cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()
print("="*60)
print("H₂ FCI 基准能量")
print("="*60)
for i, e in enumerate(E_fcis):
    exc = (e - E_fcis[0]) * 27.2114
    print(f"E{i} = {e:.8f} Ha  |  激发能: {exc:.4f} eV")

ha = nkx.operator.from_pyscf_molecule(mol)

# Hilbert 空间
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    n_fermions_per_spin=(1,1),
)
hi_ensemble = hi ** K

# 采样器 (增加 chains 和 sweep_size)
edges = [(0, 1), (2, 3)]
g = nk.graph.Graph(edges=edges)
single_rule = nk.sampler.rules.FermionHopRule(hilbert=hi, graph=g)
tensor_rule = nk.sampler.rules.TensorRule(hilbert=hi_ensemble, rules=[single_rule]*K)
sampler = nk.sampler.MetropolisSampler(hi_ensemble, rule=tensor_rule, n_chains=32, sweep_size=32)

# ==============================================================================
# 2. 神经网络 (增大容量)
# ==============================================================================
class SingleStateAnsatz(nnx.Module):
    def __init__(self, n_spin_orbitals: int, hidden_dim=32, *, rngs: nnx.Rngs):
        super().__init__()
        self.linear1 = nnx.Linear(n_spin_orbitals, hidden_dim, rngs=rngs, param_dtype=complex)
        self.linear2 = nnx.Linear(hidden_dim, hidden_dim, rngs=rngs, param_dtype=complex)
        self.output = nnx.Linear(hidden_dim, 1, rngs=rngs, param_dtype=complex)

    def __call__(self, x):
        h = nnx.tanh(self.linear1(x.astype(complex)))
        h = nnx.tanh(self.linear2(h))
        out = self.output(h)
        return jnp.squeeze(out)

class NESTotalAnsatz(nnx.Module):
    def __init__(self, n_spin_orbitals, n_states=K, hidden_dim=32):
        super().__init__()
        self.n_states = n_states
        self.n_spin_orbitals = n_spin_orbitals
        key = jax.random.key(42)
        keys = jax.random.split(key, n_states)
        self.single_ansatz_list = [
            SingleStateAnsatz(n_spin_orbitals, hidden_dim, rngs=nnx.Rngs(keys[i]))
            for i in range(n_states)
        ]

    def __call__(self, x_batch):
        K_state = self.n_states
        M = []
        for i in range(K_state):
            row = [self.single_ansatz_list[j](x_batch[i]) for j in range(K_state)]
            M.append(jnp.array(row))
        M = jnp.stack(M)
        psi_total = jnp.linalg.det(M)
        return psi_total, M

# ==============================================================================
# 3. 辅助函数
# ==============================================================================
def Ham_psi(ha, single_ansatz, x):
    x = x.squeeze()
    x_primes, mels = ha.get_conn_padded(x)
    psi_vals = jax.vmap(single_ansatz)(x_primes)
    return jnp.sum(mels * psi_vals)

def Ham_Psi(ha, total_ansatz, x_batch):
    K = total_ansatz.n_states
    H_mat = []
    for i in range(K):
        row = []
        for j in range(K):
            v = Ham_psi(ha, total_ansatz.single_ansatz_list[j], x_batch[i])
            row.append(v)
        H_mat.append(row)
    return jnp.array(H_mat, dtype=complex)

def compute_local_energy(ha, total_ansatz, x_batch, eps=1e-6):
    psi, M = total_ansatz(x_batch)
    # 检查行列式，动态调整 eps
    det_val = jnp.linalg.det(M)
    cond = jnp.abs(det_val) < 1e-4
    actual_eps = jnp.where(cond, 1e-4, eps)
    M_reg = M + actual_eps * jnp.eye(M.shape[0], dtype=M.dtype)
    Hp = Ham_Psi(ha, total_ansatz, x_batch)
    el_mat = jnp.linalg.solve(M_reg, Hp)
    return jnp.trace(el_mat).real, el_mat

compute_local_energy_batch = jax.vmap(
    compute_local_energy,
    in_axes=(None, None, 0, None),
    out_axes=(0, 0)
)

def compute_average_local_energy(ha, model, samples, eps=1e-6):
    tr_els, el_mats = compute_local_energy_batch(ha, model, samples, eps)
    tr_avg = tr_els.mean()
    el_mat_avg = el_mats.mean(axis=0)
    return tr_avg, el_mat_avg

def loss_fn(params, ha, x_batch):
    graphdef, variables = params
    model = nnx.merge(graphdef, variables)
    tr_avg, _ = compute_average_local_energy(ha, model, x_batch, eps=1e-6)
    return tr_avg

value_and_grad = jax.value_and_grad(loss_fn, argnums=0)

# ==============================================================================
# 4. forward 函数 (采样用)
# ==============================================================================
def forward(params, x_batch):
    graphdef, variables = params
    model = nnx.merge(graphdef, variables)
    n_chains = x_batch.shape[0]
    K_state = model.n_states
    n_spin = model.n_spin_orbitals
    x_reshaped = x_batch.reshape(n_chains, K_state, n_spin)
    def single_logpsi(x):
        psi, _ = model(x)
        return jnp.log(jnp.abs(psi) + 1e-8)
    log_psi_batch = jax.vmap(single_logpsi)(x_reshaped)
    return log_psi_batch

# ==============================================================================
# 5. 初始化模型、采样器、优化器
# ==============================================================================
total_ansatz = NESTotalAnsatz(n_spin_orbitals=4, n_states=K, hidden_dim=32)
graphdef, variables = nnx.split(total_ansatz)

# 采样器状态
sampler_state = sampler.init_state(forward, (graphdef, variables), seed=1)

# 优化器：AdamW + 梯度裁剪 + 余弦衰减
learning_rate = optax.cosine_decay_schedule(init_value=1e-3, decay_steps=300, alpha=0.1)
optimizer = optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.adamw(learning_rate, weight_decay=1e-5)
)
opt_state = optimizer.init(variables)

# ==============================================================================
# 6. 训练循环 (每 20 步对角化并打印)
# ==============================================================================
n_iter = 200
chain_length = 200   # 增加采样步数
loss_record = []

print("\n开始训练 NES-VMC...")
for step in tqdm(range(n_iter)):
    samples, sampler_state = sampler.sample(
        forward, (graphdef, variables), state=sampler_state, chain_length=chain_length
    )
    samples_flat = samples.reshape(-1, K, 4)
    loss_val, grads = value_and_grad((graphdef, variables), ha, samples_flat)
    loss_record.append(loss_val)
    
    grad_graph, grad_vars = grads
    updates, opt_state = optimizer.update(grad_vars, opt_state, variables)
    variables = optax.apply_updates(variables, updates)
    
    # 每 20 步计算当前对角化能量
    if step % 20 == 0:
        # 用当前模型重新计算平均能量矩阵
        curr_model = nnx.merge(graphdef, variables)
        _, el_mat_avg = compute_average_local_energy(ha, curr_model, samples_flat, eps=1e-6)
        el_mat_sym = (el_mat_avg + el_mat_avg.conj().T) / 2
        eigs = jnp.linalg.eigvalsh(el_mat_sym).real
        eigs = jnp.sort(eigs)
        print(f"\nStep {step:4d} | Loss (trace): {loss_val:.6f} Ha")
        print(f"         E0 = {eigs[0]:.6f} Ha, E1 = {eigs[1]:.6f} Ha  |  ΔE = {(eigs[1]-eigs[0])*27.2114:.2f} eV")

# 合并最终模型
total_ansatz = nnx.merge(graphdef, variables)

# ==============================================================================
# 7. 最终精确采样并输出
# ==============================================================================
print("\n最终采样，计算能量矩阵...")
final_samples, _ = sampler.sample(
    forward, (graphdef, variables), state=sampler_state, chain_length=2000
)
final_samples_flat = final_samples.reshape(-1, K, 4)
_, el_mat_avg = compute_average_local_energy(ha, total_ansatz, final_samples_flat, eps=1e-6)
el_mat_sym = (el_mat_avg + el_mat_avg.conj().T) / 2
eigen_energies = jnp.linalg.eigvalsh(el_mat_sym).real
eigen_energies = jnp.sort(eigen_energies)

print("\n" + "="*60)
print("NES-VMC 最终激发态能量 (Ha)")
print("="*60)
for i, e in enumerate(eigen_energies):
    exc = (e - eigen_energies[0]) * 27.2114
    print(f"E{i} = {e:.8f} Ha  |  激发能: {exc:.4f} eV")

In [ ]:
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import numpy as np
from pyscf import gto, scf, fci
from flax import nnx
import optax
from tqdm import tqdm

# ==============================================================================
# 1. 全局参数 & H₂ 分子
# ==============================================================================
K = 2
bond_length = 1.4
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]
mol = gto.M(atom=geometry, basis='STO-3G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)

# FCI 基准
cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()
print("="*60)
print("H₂ FCI 基准能量")
print("="*60)
for i, e in enumerate(E_fcis):
    exc = (e - E_fcis[0]) * 27.2114
    print(f"E{i} = {e:.8f} Ha  |  激发能: {exc:.4f} eV")

ha = nkx.operator.from_pyscf_molecule(mol)

# Hilbert 空间
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    n_fermions_per_spin=(1,1),
)
hi_ensemble = hi ** K

# 采样器
edges = [(0, 1), (2, 3)]
g = nk.graph.Graph(edges=edges)
single_rule = nk.sampler.rules.FermionHopRule(hilbert=hi, graph=g)
tensor_rule = nk.sampler.rules.TensorRule(hilbert=hi_ensemble, rules=[single_rule]*K)
sampler = nk.sampler.MetropolisSampler(hi_ensemble, rule=tensor_rule, n_chains=32, sweep_size=32)

# ==============================================================================
# 2. 神经网络 (hidden_dim=32)
# ==============================================================================
class SingleStateAnsatz(nnx.Module):
    def __init__(self, n_spin_orbitals: int, hidden_dim=32, *, rngs: nnx.Rngs):
        super().__init__()
        self.linear1 = nnx.Linear(n_spin_orbitals, hidden_dim, rngs=rngs, param_dtype=complex)
        self.linear2 = nnx.Linear(hidden_dim, hidden_dim, rngs=rngs, param_dtype=complex)
        self.output = nnx.Linear(hidden_dim, 1, rngs=rngs, param_dtype=complex)

    def __call__(self, x):
        h = nnx.tanh(self.linear1(x.astype(complex)))
        h = nnx.tanh(self.linear2(h))
        out = self.output(h)
        return jnp.squeeze(out)

class NESTotalAnsatz(nnx.Module):
    def __init__(self, n_spin_orbitals, n_states=K, hidden_dim=32):
        super().__init__()
        self.n_states = n_states
        self.n_spin_orbitals = n_spin_orbitals
        key = jax.random.key(42)
        keys = jax.random.split(key, n_states)
        self.single_ansatz_list = [
            SingleStateAnsatz(n_spin_orbitals, hidden_dim, rngs=nnx.Rngs(keys[i]))
            for i in range(n_states)
        ]

    def __call__(self, x_batch):
        K_state = self.n_states
        M = []
        for i in range(K_state):
            row = [self.single_ansatz_list[j](x_batch[i]) for j in range(K_state)]
            M.append(jnp.array(row))
        M = jnp.stack(M)
        psi_total = jnp.linalg.det(M)
        return psi_total, M

# ==============================================================================
# 3. 辅助函数
# ==============================================================================
def Ham_psi(ha, single_ansatz, x):
    x = x.squeeze()
    x_primes, mels = ha.get_conn_padded(x)
    psi_vals = jax.vmap(single_ansatz)(x_primes)
    return jnp.sum(mels * psi_vals)

def Ham_Psi(ha, total_ansatz, x_batch):
    K = total_ansatz.n_states
    H_mat = []
    for i in range(K):
        row = []
        for j in range(K):
            v = Ham_psi(ha, total_ansatz.single_ansatz_list[j], x_batch[i])
            row.append(v)
        H_mat.append(row)
    return jnp.array(H_mat, dtype=complex)

def compute_local_energy(ha, total_ansatz, x_batch, eps=1e-6):
    psi, M = total_ansatz(x_batch)
    det_val = jnp.linalg.det(M)
    cond = jnp.abs(det_val) < 1e-4
    actual_eps = jnp.where(cond, 1e-4, eps)
    M_reg = M + actual_eps * jnp.eye(M.shape[0], dtype=M.dtype)
    Hp = Ham_Psi(ha, total_ansatz, x_batch)
    el_mat = jnp.linalg.solve(M_reg, Hp)
    return jnp.trace(el_mat).real, el_mat

compute_local_energy_batch = jax.vmap(
    compute_local_energy,
    in_axes=(None, None, 0, None),
    out_axes=(0, 0)
)

def compute_average_local_energy(ha, model, samples, eps=1e-6):
    tr_els, el_mats = compute_local_energy_batch(ha, model, samples, eps)
    tr_avg = tr_els.mean()
    el_mat_avg = el_mats.mean(axis=0)
    return tr_avg, el_mat_avg

def loss_fn(params, ha, x_batch):
    graphdef, variables = params
    model = nnx.merge(graphdef, variables)
    tr_avg, _ = compute_average_local_energy(ha, model, x_batch, eps=1e-6)
    return tr_avg

value_and_grad = jax.value_and_grad(loss_fn, argnums=0)

# ==============================================================================
# 4. forward 函数 (采样用)
# ==============================================================================
def forward(params, x_batch):
    graphdef, variables = params
    model = nnx.merge(graphdef, variables)
    n_chains = x_batch.shape[0]
    K_state = model.n_states
    n_spin = model.n_spin_orbitals
    x_reshaped = x_batch.reshape(n_chains, K_state, n_spin)
    def single_logpsi(x):
        psi, _ = model(x)
        return jnp.log(jnp.abs(psi) + 1e-8)
    log_psi_batch = jax.vmap(single_logpsi)(x_reshaped)
    return log_psi_batch

# ==============================================================================
# 5. 初始化模型、采样器、优化器 (SGD + momentum)
# ==============================================================================
total_ansatz = NESTotalAnsatz(n_spin_orbitals=4, n_states=K, hidden_dim=32)
graphdef, variables = nnx.split(total_ansatz)
sampler_state = sampler.init_state(forward, (graphdef, variables), seed=1)

# SGD 优化器：学习率 5e-4，动量 0.9，并加入梯度裁剪
learning_rate = 5e-4
optimizer = optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.sgd(learning_rate, momentum=0.9, nesterov=False)
)
opt_state = optimizer.init(variables)

# ==============================================================================
# 6. 训练循环 (每 20 步对角化并打印)
# ==============================================================================
n_iter = 300
chain_length = 400
loss_record = []

print("\n开始训练 NES-VMC (优化器: SGD + momentum)...")
for step in tqdm(range(n_iter)):
    samples, sampler_state = sampler.sample(
        forward, (graphdef, variables), state=sampler_state, chain_length=chain_length
    )
    samples_flat = samples.reshape(-1, K, 4)
    loss_val, grads = value_and_grad((graphdef, variables), ha, samples_flat)
    loss_record.append(loss_val)
    
    grad_graph, grad_vars = grads
    updates, opt_state = optimizer.update(grad_vars, opt_state, variables)
    variables = optax.apply_updates(variables, updates)
    
    if step % 20 == 0:
        curr_model = nnx.merge(graphdef, variables)
        _, el_mat_avg = compute_average_local_energy(ha, curr_model, samples_flat, eps=1e-6)
        el_mat_sym = (el_mat_avg + el_mat_avg.conj().T) / 2
        eigs = jnp.linalg.eigvalsh(el_mat_sym).real
        eigs = jnp.sort(eigs)
        print(f"\nStep {step:4d} | Loss (trace): {loss_val:.6f} Ha")
        print(f"         E0 = {eigs[0]:.6f} Ha, E1 = {eigs[1]:.6f} Ha  |  ΔE = {(eigs[1]-eigs[0])*27.2114:.2f} eV")

total_ansatz = nnx.merge(graphdef, variables)

# ==============================================================================
# 7. 最终采样并输出
# ==============================================================================
print("\n最终采样，计算能量矩阵...")
final_samples, _ = sampler.sample(
    forward, (graphdef, variables), state=sampler_state, chain_length=2000
)
final_samples_flat = final_samples.reshape(-1, K, 4)
_, el_mat_avg = compute_average_local_energy(ha, total_ansatz, final_samples_flat, eps=1e-6)
el_mat_sym = (el_mat_avg + el_mat_avg.conj().T) / 2
eigen_energies = jnp.linalg.eigvalsh(el_mat_sym).real
eigen_energies = jnp.sort(eigen_energies)

print("\n" + "="*60)
print("NES-VMC 最终激发态能量 (Ha) - SGD 优化器")
print("="*60)
for i, e in enumerate(eigen_energies):
    exc = (e - eigen_energies[0]) * 27.2114
    print(f"E{i} = {e:.8f} Ha  |  激发能: {exc:.4f} eV")